# Type II forecasting: results on the test years (2020–2024)

Reads what `evaluate.py` produces, so nothing here needs a GPU:

```bash
python -m downstream_apps.radioburst.evaluate predict --checkpoint checkpoints/surya-....ckpt      # GPU
python -m downstream_apps.radioburst.evaluate predict --checkpoint checkpoints/baseline-....ckpt   # GPU
python -m downstream_apps.radioburst.evaluate score                                              # CPU
```

With no predictions yet, the notebook still runs and shows the reference forecasts.

In [ ]:
import os
import sys
from pathlib import Path

# Run from anywhere inside the repo: put the repo root on the path.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "workshop_infrastructure").is_dir())
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from downstream_apps.radioburst.configs import load_radioburst_config

cfg = load_radioburst_config("downstream_apps/radioburst/configs/config_script.yaml")

from downstream_apps.radioburst.evaluate import load_events, reference_forecasts, score, split_labels
from downstream_apps.radioburst.labels import LABELS

PRED_DIR = Path("predictions")  # where evaluate.py predict wrote its CSVs

## The score table

- `tss` with its rotation-bootstrap 95% interval; probabilistic forecasts are thresholded at
  the value that maximizes TSS on **val**.
- `dtss_vs_mx`: TSS minus the M/X-flare baseline's, with a **paired** interval. **This is the
  headline**: the claim "Surya beats the flare baseline" needs `dtss_ci_low > 0`.
- `bss`: Brier skill against the training-period event rate (probabilistic forecasts only).

In [ ]:
table = score(cfg, PRED_DIR, n_boot=1000)
table.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, label in zip(axes, LABELS):
    t = table[table.label == label].reset_index(drop=True)
    ax.errorbar(t.dtss_vs_mx, t.index, xerr=[t.dtss_vs_mx - t.dtss_ci_low, t.dtss_ci_high - t.dtss_vs_mx], fmt="o")
    ax.axvline(0, color="k", lw=0.8)
    ax.set_yticks(t.index, t.forecast); ax.set_title(f"{label}: TSS minus M/X-flare baseline (95% paired CI)")
plt.tight_layout()

## Reliability

Are the probabilities honest? Points on the diagonal are calibrated. Expect the models to sit
**above** the diagonal on the test years: cycle 25 has about twice cycle 24's Type II rate, so
probabilities learned on cycle 24 are too low even after the subsampling correction.

In [ ]:
events = load_events(cfg)
test = split_labels(cfg, "test", events)
models = {f.stem[: -len("_test")]: pd.read_csv(f, index_col=0, parse_dates=True) for f in sorted(PRED_DIR.glob("*_test.csv"))}

if not models:
    print(f"No predictions in {PRED_DIR.resolve()} yet: run `evaluate.py predict` first.")
else:
    bins = np.array([0, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0])
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, label in zip(axes, LABELS):
        for name, p in models.items():
            common = test.index.intersection(p.index)
            df = pd.DataFrame({"p": p.loc[common, label], "y": test.loc[common, label]})
            g = df.groupby(pd.cut(df.p, bins), observed=True).agg(p=("p", "mean"), y=("y", "mean"), n=("y", "size"))
            ax.plot(g.p, g.y, "o-", label=name)
        ax.plot([0, 1], [0, 1], "k:", lw=0.8)
        ax.axhline(test[label].mean(), color="0.6", lw=0.8, label="test event rate")
        ax.set(xscale="log", xlim=(0.005, 1), ylim=(0, 1), xlabel="forecast probability", ylabel="observed frequency", title=label)
        ax.legend(fontsize=8)
    plt.tight_layout()

## Case study: May 2024

AR 13664 produced the strongest geomagnetic storm in two decades. How did the forecasts
track the Type II onsets?

In [ ]:
window = slice("2024-04-25", "2024-06-05")
climatology = {n: 0.0 for n in LABELS}  # unused here
mx = reference_forecasts(cfg, test.loc[window], events, climatology)["mx_flare_24h"]["type2"]

fig, ax = plt.subplots(figsize=(12, 3.5))
for name, p in models.items():
    ax.plot(p.loc[window].index, p.loc[window, "type2"], label=f"{name}: P(Type II within 24 h)")
ax.fill_between(mx.index, 0, mx * 0.05, step="post", color="orange", alpha=0.4, label="M/X flare in past 24 h")
onsets = events[(events.onset >= window.start) & (events.onset <= window.stop)]
for t, ip in zip(onsets.onset, onsets.reaches_ip):
    ax.axvline(t, color="red" if ip else "k", lw=0.8, alpha=0.7)
ax.set_title("Forecast probability vs Type II onsets (black; red = reaches IP)")
ax.set_ylim(0, 1); ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()

## Before you believe any of this

- **Effective sample size.** Test has ~187 onsets but only ~43 rotations with events; the
  intervals above account for that, a row-level bootstrap would not.
- **Base-rate shift.** Train (cycle 24) has ~4.8% positive issue times, test ~8.8%. TSS and AUC
  are unaffected; Brier skill and reliability are not.
- **Label noise.** 34 onsets come from behind-limb flares SDO cannot see
  (`data.ds_exclude_behind_limb` tests their effect). "Reaches IP" depends on where the
  catalog records the burst's end, which data gaps can shift.
- **The interplanetary label.** The frame precedes the eruption, so the CME speed that decides
  whether a shock reaches IP is not in the input. Similar skill on both labels means the model
  learned "eruptive region on the disk", not "IP-capable shock".
- **Operational use** would need near-real-time SDO data, calibrated differently from the
  science data Surya was trained on.